# Optimistic Commits, Conflicts and Metadata Growth

Companion notebook for the [ACID on Object Storage](https://codemia.io/courses/big_data_engines/bd_acid_on_object_storage) lesson on Codemia.

**What you will build**: a discrete event simulator for optimistic concurrency on a table format: writers prepare files, then
race to swap the table pointer. You will watch throughput collapse as writers are added, find out why
a long running job can starve behind a one minute stream, measure what partition level conflict
detection buys, and price the metadata a committing stream leaves behind.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import heapq

import numpy as np
import matplotlib.pyplot as plt

PARTITIONS = 200          # the table is partitioned by day, about half a year of them
COMMIT_S = 1.8            # write new metadata, then one atomic compare-and-swap
HORIZON_S = 3_600         # simulate an hour

print(f'table: {PARTITIONS} partitions')
print(f'a commit takes {COMMIT_S} s of metadata work and is atomic, '
      f'so commits serialise')
print(f'simulating {HORIZON_S // 60} minutes of writers')

## Concept recap

Object storage gives you one guarantee that matters here: a single object write is atomic, and a
catalog can swap a pointer from one metadata file to another atomically. There is no rename, no
locking, and no way to update several files at once. Every table format on object storage, Iceberg,
Delta and Hudi alike, is built on that one primitive.

The pattern is optimistic concurrency. A writer reads the current snapshot, writes new data files
into place, which nobody can see yet because no metadata points at them, and then tries to swap the
table pointer from version N to version N plus one. If somebody else swapped first, the writer's
version number is stale, the swap fails, and the writer has to decide whether its work is still
valid against the newer snapshot.

That means two different costs. Conflicts, which waste whole rewrites of data that was already
written, and metadata, because every successful commit leaves another version of the table behind
forever until something cleans it up. Both are simulated below.

## A commit simulator

In [ ]:
def simulate(writers, prepare_s=45.0, overlap_parts=4, detection='table',
             horizon=HORIZON_S, revalidate_frac=1.0, backoff_s=0.0, seed=0, jitter=0.25,
             specs=None):
    """Each writer loops: read a snapshot, prepare files, attempt the swap, retry on conflict.

    detection='table' conflicts with any commit since the snapshot, which is what a
    blind append-and-overwrite does. detection='partition' conflicts only when the
    partitions overlap, which is what a format with partition level validation can do.
    revalidate_frac is how much of the prepare work a retry has to redo. Pass specs as
    a list of {'prepare', 'parts', 'pool'} to mix slow and fast writers on one table.
    """
    rng = np.random.default_rng(seed)
    if specs is None:
        specs = [{'prepare': prepare_s, 'parts': overlap_parts,
                  'pool': np.arange(PARTITIONS)} for _ in range(writers)]
    writers = len(specs)
    log = []                      # committed (time, partitions)
    events = []
    state = {}
    for w in range(writers):
        start = float(rng.uniform(0, specs[w]['prepare']))   # writers are not synchronised
        heapq.heappush(events, (start, w, 'read'))
    cas_free = 0.0
    commits = attempts = conflicts = 0
    wasted_s = 0.0
    latencies = []
    per_writer = np.zeros(writers, int)
    tries_by_writer = np.zeros(writers, int)
    wasted_by_writer = np.zeros(writers)

    while events:
        t, w, phase = heapq.heappop(events)
        if t > horizon:
            break
        if phase == 'read':
            spec = specs[w]
            k = min(spec['parts'], len(spec['pool']))
            chosen = set(rng.choice(spec['pool'], k, replace=False).tolist())
            chosen |= set(spec.get('always', ()))      # partitions this writer always rewrites
            parts = frozenset(chosen)
            work = float(spec['prepare'] * rng.lognormal(0, jitter))
            state[w] = {'snapshot_t': t, 'parts': parts, 'work': work,
                        'first': t, 'attempt': 1}
            heapq.heappush(events, (t + work, w, 'commit'))
        else:
            st = state[w]
            attempts += 1
            tries_by_writer[w] += 1
            clash = [c for c in log if c[0] > st['snapshot_t']
                     and (detection == 'table' or c[1] & st['parts'])]
            if clash:
                conflicts += 1
                wasted_s += st['work']
                wasted_by_writer[w] += st['work']
                redo = st['work'] * revalidate_frac
                st['snapshot_t'] = t
                st['attempt'] += 1
                st['work'] = redo
                delay = redo + backoff_s * (2 ** min(st['attempt'] - 2, 5)) * rng.random()
                heapq.heappush(events, (t + delay, w, 'commit'))
            else:
                begin = max(t, cas_free)               # the swap itself is serialised
                cas_free = begin + COMMIT_S
                log.append((cas_free, st['parts']))
                commits += 1
                per_writer[w] += 1
                latencies.append(cas_free - st['first'])
                heapq.heappush(events, (cas_free, w, 'read'))

    return {'commits': commits, 'attempts': attempts, 'conflicts': conflicts,
            'per_hour': commits * 3600 / horizon, 'wasted_s': wasted_s,
            'amplification': attempts / commits if commits else float('inf'),
            'p50': float(np.median(latencies)) if latencies else float('nan'),
            'p95': float(np.percentile(latencies, 95)) if latencies else float('nan'),
            'starved': int((per_writer == 0).sum()), 'per_writer': per_writer,
            'tries_by_writer': tries_by_writer,
            'wasted_by_writer': wasted_by_writer}


for writers in [1, 4]:
    r = simulate(writers=writers, prepare_s=45)
    print(f'{writers} writer(s), 45 s per attempt, table level detection')
    print(f'  commits in the hour  {r["commits"]:>6}')
    print(f'  attempts             {r["attempts"]:>6}  '
          f'({r["amplification"]:.2f} attempts per commit)')
    print(f'  conflicts            {r["conflicts"]:>6}')
    print(f'  wasted prepare time  {r["wasted_s"] / 3600:>6.1f} cluster hours')
    print(f'  commit latency p50   {r["p50"]:>6.1f} s, p95 {r["p95"]:.1f} s')
    print()

One writer is the uninteresting case and the useful baseline: every attempt lands, nothing is wasted,
and a commit takes as long as it took to prepare. That is the regime every table format demo runs in.

Four writers is where it starts to get expensive. The commit count barely moves, but it now takes
three attempts to land one commit, and the writers between them throw away roughly three cluster
hours of prepare work inside a single wall clock hour. Nothing failed. Every job logged a retry and
carried on.

So the question is what adding writers actually buys, because optimistic concurrency does not share a
table the way a database does.

## Adding writers adds no throughput

In [ ]:
REPEATS = 4          # average over several arrival patterns, not one seed's luck

def sweep(w, **kw):
    runs = [simulate(writers=w, prepare_s=45, seed=100 * k + w, **kw) for k in range(REPEATS)]
    keys = ['per_hour', 'amplification', 'wasted_s', 'p50', 'p95', 'starved']
    return {k: float(np.mean([r[k] for r in runs])) for k in keys}


print(f'{"writers":>9}{"commits/h":>12}{"attempts/commit":>18}{"wasted":>16}'
      f'{"p50 latency":>14}{"p95 latency":>14}{"starved":>9}')
rows = []
for w in [1, 2, 4, 8, 12, 16, 24, 32]:
    r = sweep(w)
    rows.append((w, r))
    print(f'{w:>9}{r["per_hour"]:>12.0f}{r["amplification"]:>18.2f}'
          f'{r["wasted_s"] / 3600:>12.1f} h{r["p50"]:>12.1f} s{r["p95"]:>12.1f} s'
          f'{r["starved"]:>9.1f}')

one, last = rows[0][1], rows[-1][1]
print()
print(f'one writer alone:  {one["per_hour"]:.0f} commits/hour, '
      f'{one["wasted_s"] / 3600:.1f} wasted hours')
print(f'{rows[-1][0]} writers together: {last["per_hour"]:.0f} commits/hour, '
      f'{last["wasted_s"] / 3600:.1f} wasted hours, '
      f'{last["starved"]:.1f} writers that never committed at all')
print(f'the serial ceiling, 1 / (prepare + commit): '
      f'{3600 / (45 + COMMIT_S):.0f} commits/hour')

Throughput does not rise at all. Thirty two writers land about as many commits per hour as one
writer does, because with table level conflict detection every successful commit invalidates every
other attempt in flight, so the table can only ever progress one prepare at a time. The last line is
that ceiling: one divided by the time to prepare and commit once.

Everything else, though, gets dramatically worse. Attempts per commit grows linearly with the writer
count, the wasted prepare time passes thirty cluster hours per wall clock hour, the p95 latency grows
by a factor of thirty, and a third of the writers never commit anything during the hour. In a real
system this does not look like an error. Every job logs that it retried, the table keeps accepting
commits, and only the bill and the lag notice.

In [ ]:
ws = np.array([w for w, _ in rows])
tput = np.array([r['per_hour'] for _, r in rows])
waste = np.array([r['wasted_s'] / 3600 for _, r in rows])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
ax1.plot(ws, tput, 'o-', color='tab:blue', label='commits per hour')
ax1.set_xlabel('concurrent writers')
ax1.set_ylabel('commits per hour', color='tab:blue')
ax1.grid(alpha=0.3)
ax1b = ax1.twinx()
ax1b.plot(ws, waste, 's--', color='tab:red', label='wasted prepare (hours)')
ax1b.set_ylabel('wasted prepare work (hours per hour)', color='tab:red')
ax1.set_title('More writers, less throughput, much more waste')

for detection, colour in [('table', 'tab:red'), ('partition', 'tab:green')]:
    y = [sweep(w, detection=detection)['per_hour'] for w in ws]
    ax2.plot(ws, y, 'o-', color=colour, label=f'{detection} level detection')
ax2.plot(ws, [3600 / (45 + COMMIT_S) * w for w in ws], ':', color='black',
         label='if writers never conflicted')
ax2.set_xlabel('concurrent writers')
ax2.set_ylabel('commits per hour')
ax2.set_yscale('log')
ax2.set_title('Conflict granularity decides whether a table scales')
ax2.legend()
ax2.grid(alpha=0.3)
plt.show()

The right panel is the design rule, and it is about granularity rather than tuning. With table level
detection the line is flat: the table is a single writer no matter how many jobs you point at it.
With partition level detection it climbs with the writer count, because writers working on different
partitions have nothing to disagree about and all of their swaps succeed. Neither line reaches the
dotted one, because the swap itself serialises, which is a limit you reach at thousands of commits an
hour rather than tens.

That is why the standard advice for a lakehouse table is one writer per partition set, and why a
format's conflict validation granularity matters more than any setting it exposes.

## Partition level detection, and who starves

In [ ]:
print('16 writers, 45 s per attempt, each touching 4 of 200 partitions')
print()
print(f'{"detection":>12}{"commits/h":>12}{"attempts/commit":>18}'
      f'{"wasted":>12}{"p95 latency":>14}')
for detection in ['table', 'partition']:
    r = simulate(writers=16, prepare_s=45, detection=detection, seed=3)
    print(f'{detection:>12}{r["per_hour"]:>12.0f}{r["amplification"]:>18.2f}'
          f'{r["wasted_s"]:>10,.0f} s{r["p95"]:>12.1f} s')

print()
print('the same two, when every writer touches the same hot partition set:')
print(f'{"detection":>12}{"commits/h":>12}{"attempts/commit":>18}'
      f'{"wasted":>12}{"p95 latency":>14}')
for detection in ['table', 'partition']:
    r = simulate(writers=16, prepare_s=45, detection=detection,
                 overlap_parts=PARTITIONS, seed=3)
    print(f'{detection:>12}{r["per_hour"]:>12.0f}{r["amplification"]:>18.2f}'
          f'{r["wasted_s"]:>10,.0f} s{r["p95"]:>12.1f} s')

Partition level validation is close to free and it removes the conflict entirely when writers work on
different parts of the table, which is the common case: a backfill rewriting last March while a
stream appends to today has no genuine disagreement, and only a coarse conflict check turns that into
a fight. A format that validates at partition or file level lets both land.

The second block is the honest limit, and the two rows are identical by construction: when every
writer touches every partition, partition level validation is asking the same question as table level
validation and gets the same answer. The conflicts are real. No amount of cleverness in the metadata
layer removes a genuine disagreement about the same files, and the answer is fewer writers: one job
that owns those partitions, with everyone else feeding it.

In [ ]:
OLD_PARTITIONS = np.arange(0, 60)        # last quarter, what a backfill rewrites
TODAY = np.array([PARTITIONS - 1])       # what a stream appends to


def mixed(backfill_prepare, detection='table', backfill_pool=None, hits_today=False,
          horizon=7_200):
    specs = [{'prepare': backfill_prepare, 'parts': 20,
              'pool': backfill_pool if backfill_pool is not None else np.arange(PARTITIONS),
              'always': TODAY.tolist() if hits_today else []},
             {'prepare': 30.0, 'parts': 1, 'pool': TODAY}]
    return simulate(None, detection=detection, horizon=horizon, seed=9, specs=specs)


print('two hours of a backfill sharing one table with a stream that commits every 30 s')
print('(the backfill rewrites 20 partitions chosen anywhere in the table)')
print()
print(f'{"backfill prepare":>18}{"backfill commits":>18}{"its attempts":>14}'
      f'{"stream commits":>16}{"backfill waste":>17}')
for prep in [20, 60, 300, 1_200]:
    r = mixed(prep)
    print(f'{prep:>16} s{r["per_writer"][0]:>18}{r["tries_by_writer"][0]:>14}'
          f'{r["per_writer"][1]:>16}{r["wasted_by_writer"][0] / 3600:>14.1f} h')

print()
print('the same twenty minute backfill under three conflict rules:')
cases = [
    ('table level, old partitions only', dict(detection='table',
                                              backfill_pool=OLD_PARTITIONS)),
    ('partition level, old partitions', dict(detection='partition',
                                             backfill_pool=OLD_PARTITIONS)),
    ("partition level, also rewrites today", dict(detection='partition',
                                                  backfill_pool=OLD_PARTITIONS,
                                                  hits_today=True)),
]
for label, kw in cases:
    r = mixed(1_200, **kw)
    print(f'  {label:>38}: backfill {r["per_writer"][0]:>2} commits '
          f'in {r["tries_by_writer"][0]:>3} attempts, '
          f'stream {r["per_writer"][1]:>3} commits')

Starvation is the part of optimistic concurrency that surprises people, and it is absolute rather
than gradual. The protocol has no fairness property at all: whoever finishes preparing first wins the
swap. Any writer slower than the stream's commit interval is racing against at least one commit every
time it tries, so it loses every race it enters. The sixty second job, only twice the stream's
interval, committed nothing in two hours. Neither did the twenty minute job. Both spent the whole
window retrying, and neither logged an error.

The first row is the same rule seen from the other side. A twenty second writer is faster than the
stream's interval, so it wins, and it wins so consistently that the stream drops from two hundred and
twenty commits to forty four. Priority under optimistic concurrency is just speed, whether or not
that is the priority you wanted.

The three conflict rules at the bottom are the fix and its limit. Partition level validation lets the
backfill land every attempt, because rewriting last quarter and appending to today are not a real
disagreement. The third row is a backfill that also rewrites today's partition, and it starves again
at exactly the same rate as the coarse check, because now the disagreement is real. Narrow the
conflict check if the work is genuinely disjoint; otherwise shorten the commits or remove a writer.

## What the commits leave behind

In [ ]:
GET_MS = 22          # object storage GET latency for a small metadata file
PLAN_PARALLEL = 32   # concurrent metadata reads during query planning
MANIFEST_KB = 90
DATA_FILE_MB = 64


def metadata_state(days, commit_interval_s, files_per_commit=4, retain_days=7,
                   rewrite_manifests_every=None):
    """Accumulated snapshots, manifests and planning cost for a committing stream."""
    commits_per_day = 86_400 / commit_interval_s
    total_commits = commits_per_day * days
    live_snapshots = commits_per_day * min(days, retain_days)
    manifests = total_commits * 1            # one new manifest per commit
    if rewrite_manifests_every:
        manifests = min(manifests, rewrite_manifests_every / commit_interval_s)
    plan_s = manifests * (GET_MS / 1000) / PLAN_PARALLEL
    return {'commits': total_commits, 'live_snapshots': live_snapshots,
            'manifests': manifests, 'metadata_mb': total_commits * MANIFEST_KB / 1024,
            'plan_s': plan_s,
            'data_files': total_commits * files_per_commit,
            'data_gb': total_commits * files_per_commit * DATA_FILE_MB / 1024}


print('a stream appending to one table for 30 days, no maintenance:')
print(f'{"commit every":>14}{"commits":>11}{"manifests":>12}{"metadata":>12}'
      f'{"planning":>11}{"data files":>12}')
for interval in [600, 60, 30, 10]:
    m = metadata_state(30, interval)
    print(f'{interval:>12} s{m["commits"]:>11,.0f}{m["manifests"]:>12,.0f}'
          f'{m["metadata_mb"]:>10,.0f} MB{m["plan_s"]:>9.1f} s{m["data_files"]:>12,.0f}')

print()
print('the same table with manifests rewritten hourly:')
print(f'{"commit every":>14}{"manifests":>12}{"planning":>11}'
      f'{"was":>11}{"live snapshots at 7 days":>26}')
for interval in [600, 60, 30, 10]:
    raw = metadata_state(30, interval)
    tidy = metadata_state(30, interval, rewrite_manifests_every=3_600)
    print(f'{interval:>12} s{tidy["manifests"]:>12,.0f}{tidy["plan_s"]:>9.2f} s'
          f'{raw["plan_s"]:>9.1f} s{tidy["live_snapshots"]:>26,.0f}')
print()
print('without maintenance the planning cost grows with the age of the table;')
print('with hourly rewrites it depends only on the commit interval, and stops growing.')

Every commit is cheap and every commit is permanent. A stream committing every ten seconds produces a
quarter of a million snapshots in a month, each one a metadata file that the catalog keeps and each
one pinning the data files it referenced. Query planning reads the current snapshot's manifests, so
the planning time grows with the number of manifests, and a table nobody has maintained can take
longer to plan a query than to run it.

Manifest rewriting and snapshot expiry are the two maintenance jobs that fix this, and the second
table is the reason to schedule them before you need them. They are not optimisations, they are the
difference between a table that works at ten second commits and one that does not.

In [ ]:
print('why storage keeps growing after a delete, with 7 day snapshot retention:')
DAILY_WRITE_GB = 400
DAILY_DELETE_GB = 350            # a GDPR delete job, or a daily overwrite of yesterday

print(f'{"day":>5}{"logical size":>15}{"physical size":>16}{"overhead":>11}')
logical = 0.0
for day in range(1, 15):
    logical += DAILY_WRITE_GB - DAILY_DELETE_GB
    # every file written in the retention window is still referenced by some snapshot
    physical = logical + min(day, 7) * DAILY_DELETE_GB
    print(f'{day:>5}{logical:>12,.0f} GB{physical:>13,.0f} GB'
          f'{physical / max(logical, 1):>10.1f}x')

print()
print('the overhead settles at a constant amount of dead data, not a constant multiple,')
print('so it hurts most on a small table that churns:')
print(f'  retention 7 days  -> {7 * DAILY_DELETE_GB:,} GB of tombstoned data held open')
print(f'  retention 1 day   -> {1 * DAILY_DELETE_GB:,} GB')
print(f'  retention 30 days -> {30 * DAILY_DELETE_GB:,} GB')
print()
print('time travel is a storage decision. that is the decision.')

This is the invoice surprise that follows a table format migration. Time travel and rollback are
implemented by keeping the old files, so a delete does not free anything until the snapshots that
referenced those files expire. A table that overwrites most of itself daily and keeps a week of
history carries roughly a week of overwrites as dead weight, permanently.

None of that is a defect. It is the same mechanism that lets you roll back a bad write in one
command, and it is worth paying for. It is just worth paying for deliberately, with a retention
window chosen from how far back anyone has ever actually needed to travel rather than left at
whatever the format's default is.

## What to take away

- With table level conflict detection, a table is a single writer no matter how many jobs point at
  it. Thirty two writers landed the same number of commits per hour as one, while throwing away
  thirty cluster hours of prepare work and leaving a third of the writers with nothing committed.
- Partition or file level validation is what turns that flat line into one that scales, because
  writers on different partitions have nothing to disagree about.
- The protocol has no fairness, only speed. Any writer slower than the commit interval of its
  neighbours loses every race it enters, retries silently, and never lands anything.
- Partition or file level conflict validation is what makes a backfill and a stream coexist. When the
  writers genuinely touch the same partitions, no validation scheme helps and the answer is fewer
  writers.
- Every commit leaves a snapshot and a manifest behind. Planning cost grows with manifests, so
  manifest rewriting and snapshot expiry are prerequisites for frequent commits rather than
  optimisations.
- Deleted data is not freed until its snapshots expire, so the retention window is a direct
  multiplier on storage. Choose it from how far back anyone has needed to travel.